# `setup_teste` — smoke test dos fluxos

**Objetivo:** provar que **cada fluxo roda** e que **de fato gravou o que deveria** na base Parquet.
Não é carga de dados: cada bloco usa o **menor período possível** (1 pregão) e, onde dá, um
**`--limit`** — a meta é terminar o mais rápido possível.

**Cada bloco:** roda o fluxo → confere no `.db` → imprime `[OK]` ou `[VAZIO]`.
Nada aborta: um bloco vermelho não impede os seguintes. A última célula resume tudo.

**Ordem dos notebooks:** `setup_teste` (aqui) → `setup_inicial` (histórico) → `run_secundario` (dia a dia).

⚠️ **Rode a partir da pasta `code/`.** Pode dar `Run All`.

## Config — rodar primeiro
Cria o `.db` (idempotente, não apaga nada), define o dia de teste e o helper `checar()`.

In [ ]:
import sys
import time
from datetime import date
from pathlib import Path

if not (Path.cwd() / "codigos").exists():
    raise SystemExit(f"Rode a partir da pasta code/. cwd atual: {Path.cwd()}")
sys.path.insert(0, str(Path.cwd() / "Helpers"))    # p/ 'import dados, config, ...'
                                                   # (pipeline_core tambem vive la)
import pipeline_core as pc
import dados

# ==================== EDITE AQUI (opcional) ====================
LIMITE_TRADES  = 40   # calc_taxa: quantos trades processar (prioriza os que batem em API)
LIMITE_TICKERS = 5    # anbima_data: quantos tickers raspar
WORKERS        = 8    # threads das chamadas de API
LIMITE_VALID   = 10   # validar_calc_b3: quantos ativos da fila conferir
# ===============================================================

# Insumos da calculadora de renda fixa (bancos proprios, ver lib/calc.py)
DB_IPCA = "files/Database/ipca.db"
DB_DI   = "files/Database/di.db"

# D = ultimo dia util ANTES de hoje (garante que a fonte ja publicou).
# Tudo neste notebook roda em D, e so em D.
D = pc.DiaUtilAnterior(date.today()).isoformat()

print("Base:", Path(dados.Raiz()).resolve())
print("Dia de teste D =", D)
print(f"Limites: calc_taxa={LIMITE_TRADES} trades | anbima_data={LIMITE_TICKERS} tickers")

resultadosSmoke = []

def Checar(desc, sql, params=(), db=None):
    """Contagem simples: gravou alguma coisa?

    Sem `db`, consulta a base Parquet pelo DuckDB. Com `db`, cai no sqlite3 — e o
    caminho do ipca.db/di.db, que seguem SQLite por serem contrato com a
    calculadora."""
    try:
        if db:
            import sqlite3
            n = sqlite3.connect(db).execute(sql, params).fetchone()[0]
        else:
            n = dados.Escalar(sql, params or None)
    except Exception as e:
        print(f"[ERRO ] {desc}: {e}")
        resultadosSmoke.append((desc, False))
        return False
    ok = bool(n and n > 0)
    print(f"{'[OK]    ' if ok else '[VAZIO] '}{desc}: {n:,} linha(s)")
    resultadosSmoke.append((desc, ok))
    return ok

## Scraping — 1 bloco por fonte
Todos em `D` (1 pregão). Os blocos de **cálculo** vêm depois porque dependem destes.

In [ ]:
# 1. Boletim B3 (negocios) -> NegociosBrutos
pc.Boletim(D)
Checar("boletim -> NegociosBrutos",
       "SELECT COUNT(*) FROM NegociosBrutos WHERE dtNegocio = ?", (D,))

In [ ]:
# 2. Cadastro + fluxo pela B3 (getBondDetails) -> InfoAtivos + FluxoAtivos.  FONTE PRIMARIA.
#    Roda DEPOIS do boletim (precisa saber o que negociou) e ANTES do anbima_data (que so
#    preenche o que a B3 nao cobriu). Marca stFluxoValidado = 1: o fluxo da B3 E a fonte.
pc.BondDetails(D, D)
Checar("b3_bond_details -> InfoAtivos.cdFonteCadastro='B3'",
       "SELECT COUNT(*) FROM InfoAtivos WHERE cdFonteCadastro = 'B3'")
Checar("b3_bond_details -> vrAniversario (so IPCA)",
       "SELECT COUNT(*) FROM InfoAtivos WHERE vrAniversario IS NOT NULL")


In [ ]:
# 3. Anbima debentures (taxa indicativa) -> AnbimaIndicativos
pc.AnbimaDeb(D)
Checar("anbima_deb -> AnbimaIndicativos",
       "SELECT COUNT(*) FROM AnbimaIndicativos WHERE dtReferencia = ?", (D,))

In [ ]:
# 4. Anbima CRI/CRA (taxa indicativa, Playwright) -> AnbimaIndicativos
#    Como o bloco 2 ja gravou deb nesta data, o check filtra so CRI/CRA:
#    CRA* ou ticker de codigo numerico (padrao dos CRIs).
pc.AnbimaCriCra(D)
Checar("anbima_cricra -> AnbimaIndicativos (so CRI/CRA)",
       "SELECT COUNT(*) FROM AnbimaIndicativos "
       "WHERE dtReferencia = ? AND (cdTicker LIKE 'CRA%' OR cdTicker GLOB '[0-9]*')", (D,))

In [ ]:
# 5. FI Analytics planilha (caracteristicas, Playwright + login) -> InfoAtivos
#    Sem data — baixa o snapshot atual. Check: gravou linhas HOJE?
pc.FiAnalytics()
Checar("fianalytics -> InfoAtivos (gravado hoje)",
       "SELECT COUNT(*) FROM InfoAtivos WHERE DATE(dtAtualizacao) = DATE('now','localtime')")

In [ ]:
# 6. Anbima Data (caracteristicas + agenda, Playwright) -> InfoAtivos + FluxoAtivos
#    So LIMITE_TICKERS tickers negociados em D, com --force (ignora cache -> prova que raspa mesmo).
#    Check: FluxoAtivos gravado HOJE (so este fluxo escreve nessa tabela).
pc.AnbimaData(D, limit=LIMITE_TICKERS, force=True)
Checar("anbima_data -> FluxoAtivos (gravado hoje)",
       "SELECT COUNT(*) FROM FluxoAtivos WHERE DATE(dtAtualizacao) = DATE('now','localtime')")

In [ ]:
# 7. Anbima NTN-B (MtM) -> MtmAnbima. Duration em paralelo + skip do ja calculado.
pc.Ntnb(D, workers=WORKERS)
Checar("ntnb -> MtmAnbima (NTN-B)",
       "SELECT COUNT(*) FROM MtmAnbima WHERE cdTicker LIKE 'NTN-B%' AND dtReferencia = ?", (D,))

In [ ]:
# 8. Curva DI B3 -> MtmAnbima (contratos DI1, p/ o relatorio) + di.db/CurvaDi (curva
#    inteira, p/ a calculadora). Um download, dois destinos.
pc.CurvaDi(D)
Checar("curva_di -> MtmAnbima (DI1)",
       "SELECT COUNT(*) FROM MtmAnbima WHERE cdTicker LIKE 'DI1%' AND dtReferencia = ?", (D,))
Checar("curva_di -> di.db/CurvaDi (curva completa)",
       "SELECT COUNT(*) FROM CurvaDi WHERE dtReferencia = ?", (D,), db=DB_DI)


In [ ]:
# 9. Outstanding via Bloomberg -> Outstanding.  *** SO RODA NO BANCO ***
#    No PC pessoal (sem terminal Bloomberg) da [FALHA]/[VAZIO] — isso e esperado.
pc.Outstanding(D)
Checar("outstanding -> Outstanding (so no banco)",
       "SELECT COUNT(*) FROM Outstanding WHERE dtOutstanding = ?", (D,))

In [ ]:
# 10. IPCA realizado (IBGE/SIDRA) -> ipca.db/IPCA. Insumo da calculadora (VNA de IPCA+).
pc.IpcaIbge()
Checar("ipca_ibge -> IPCA (numero-indice)",
       "SELECT COUNT(*) FROM IPCA WHERE vrIndiceIPCA IS NOT NULL", db=DB_IPCA)


In [ ]:
# 11. Projecao de IPCA (Anbima) -> ipca.db/IPCAProjetado.  *** RODAR ANTES DAS 17h30 ***
#     Sem ela nao se precifica IPCA+ no mes corrente. Check: a serie cobre hoje?
pc.IpcaProjetado()
Checar("ipca_projetado -> IPCAProjetado (cobre hoje)",
       "SELECT COUNT(*) FROM IPCAProjetado WHERE dtIPCAProjetado >= ?",
       (date.today().isoformat(),), db=DB_IPCA)


In [ ]:
# 12. DI realizado (BCB/SGS) -> di.db/DiHistorico. Incremental desde o ultimo dia gravado.
#     Check: tem DI recente? (o BCB publica o DI de D so na noite de D, dai a folga)
pc.DiBcb()
Checar("di_bcb -> DiHistorico (dado recente)",
       "SELECT COUNT(*) FROM DiHistorico WHERE dtReferencia >= date(?, '-5 days')",
       (D,), db=DB_DI)


## Cálculo — 1 bloco por fluxo
Tudo na liquidação `D`. O `calc_taxa` roda com `--limit`, só o suficiente pra exercitar a cascata
FI Analytics → B3 — por isso as contagens abaixo são pequenas **de propósito**.

In [ ]:
# 13. Taxa por trade (cascata FI Analytics -> B3) -> NegociosProcessados
#    --limit prioriza trades SEM vrTaxaNegocio, ou seja: forca a cascata a rodar de verdade.
pc.CalcTaxa(D, workers=WORKERS, limit=LIMITE_TRADES, force=True)
Checar("calc_taxa -> NegociosProcessados (liquidacao D)",
       "SELECT COUNT(*) FROM NegociosProcessados WHERE dtLiquidacao = ?", (D,))
Checar("calc_taxa -> vrTaxaCalculada preenchida",
       "SELECT COUNT(*) FROM NegociosProcessados "
       "WHERE dtLiquidacao = ? AND vrTaxaCalculada IS NOT NULL", (D,))

In [ ]:
# 14. Filtrar (VALIDO / FUNDO / BROKER / PF) -> NegociosProcessados.cdStatus
pc.Filtrar(D)
Checar("filtrar -> cdStatus preenchido",
       "SELECT COUNT(*) FROM NegociosProcessados WHERE dtLiquidacao = ? AND cdStatus IS NOT NULL", (D,))

In [ ]:
# 15. Spread Anbima das indicativas -> AnbimaIndicativos.vrSpreadAnbima
pc.SpreadAnbima(D)
Checar("spread_anbima -> vrSpreadAnbima preenchido",
       "SELECT COUNT(*) FROM AnbimaIndicativos "
       "WHERE dtReferencia = ? AND vrSpreadAnbima IS NOT NULL", (D,))

In [ ]:
# 16. Match de referencia (global, sem data) -> InfoAtivos.cdReferencia
pc.MatchRef()
Checar("match_ref -> cdReferencia preenchido",
       "SELECT COUNT(*) FROM InfoAtivos WHERE cdReferencia IS NOT NULL")

In [ ]:
# 17. Spread over dos trades (casado por dtNegocio) -> NegociosProcessados.vrSpreadOver
pc.SpreadOver(D)
Checar("spread_over -> vrSpreadOver preenchido",
       "SELECT COUNT(*) FROM NegociosProcessados "
       "WHERE dtLiquidacao = ? AND vrSpreadOver IS NOT NULL", (D,))

In [ ]:
# 18. Gerar relatorio HTML (le a base inteira). Check: o arquivo foi REGRAVADO agora?
alvo  = Path("files/relatorios/relatorio_secundario.html")
antes = alvo.stat().st_mtime if alvo.exists() else 0
pc.Relatorio()
ok = alvo.exists() and alvo.stat().st_mtime > antes
resultadosSmoke.append(("relatorio -> HTML regravado", ok))
if alvo.exists():
    idade = time.time() - alvo.stat().st_mtime
    print(f"{'[OK]    ' if ok else '[VAZIO] '}relatorio: {alvo} "
          f"({alvo.stat().st_size / 1e6:.1f} MB, gravado ha {idade:.0f}s)")
else:
    print("[VAZIO] relatorio: arquivo nao foi criado")

## Resumo

In [ ]:
oks = sum(1 for _, ok in resultadosSmoke if ok)
print(f"{'#' * 62}\n# SMOKE TEST: {oks}/{len(resultadosSmoke)} conferencias OK\n{'#' * 62}")
for desc, ok in resultadosSmoke:
    print(f"  {'OK   ' if ok else 'VAZIO'}  {desc}")

faltou = [d for d, ok in resultadosSmoke if not ok]
if faltou:
    print("\nVAZIO = o fluxo rodou mas nao gravou o que se esperava. Investigar:")
    for d in faltou:
        print("  -", d)
    print("\nObs: 'outstanding' fica VAZIO fora do banco (sem Bloomberg) — normal.")
else:
    print("\nTudo certo — todos os fluxos gravaram no .db. Pode ir pro setup_inicial.ipynb.")